# 2. 细胞类型注释

基于 `0_integrate_two_datasets` 产出的 `adata1.h5ad`（130k cells, Harmony 整合, Leiden 聚类）

流程：
1. 加载数据，检查 Leiden 聚类
2. 已知 GBM marker 可视化 → 初步判断各 cluster 身份
3. `rank_genes_groups` → 验证 cluster 特异性 marker
4. 手动注释 → 保存 `adata2.h5ad`

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

sc.set_figure_params(dpi=100, color_map='viridis_r', transparent=False, frameon=False)
%config InlineBackend.figure_format = 'retina'

WRITE_DIR = Path('./write/0_integrate_two_datasets')
FIG_DIR   = Path('./figures/2_marker_gene')
FIG_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(FIG_DIR)

print(f'scanpy {sc.__version__}')

## 1. 加载数据

In [ ]:
adata = sc.read(WRITE_DIR / 'adata1.h5ad')

if 'log1p' in adata.uns:
    adata.uns['log1p']['base'] = None

adata

In [ ]:
# 检查基本分布
print('Leiden clusters:', adata.obs['leiden'].nunique())
print(adata.obs['leiden'].value_counts().sort_index())
print()
print('Tissue:', adata.obs['tissue_type'].value_counts())
print()
print('Dataset:', adata.obs['dataset'].value_counts())

In [ ]:
sc.pl.umap(adata, color=['leiden', 'tissue_type', 'dataset'],
           ncols=3, legend_loc='on data', legend_fontsize=6)

## 2. 已知 GBM Marker 基因可视化

In [ ]:
# GBM marker 基因 v3 — 修正 CDC 过标记问题
gbm_markers = {
    # --- Tumor & Glia ---
    "Tumor":          ["GFAP", "PTPRZ1", "SCG3", "C1orf61", "SOX2", "SOX6", "BCAN", "NOVA1", "NRCAM", "TUBB2B"],
    "Prol.tumor":     ["SOX2", "SOX6", "MKI67", "TOP2A", "CCDC34", "PBK", "UBE2T", "CENPV", "CENPF"],
    "Oligodendrocyte":["MAG", "CLDN11", "APLP1", "TMEM144", "CNDP1", "EDIL3", "LARP6", "MOG", "AMER2", "TUBB4A"],
    "Neuronal":       ["SYT1", "STMN2", "RBFOX3", "GRIN1", "SLC17A7", "GAD1", "GAD2"],
    # --- Lymphoid ---
    "NK/T":           ["CD3D", "CD3E", "TRBC2", "CD3G", "CD2", "TRAC", "IL7R", "TRBC1", "LCK", "SKAP1", "CD48"],
    # --- Vascular / Stromal ---
    "Pericyte":       ["COL1A2", "COL6A2", "PDGFRB", "COL3A1", "EDNRA", "LUM", "COL1A1", "PLAC9", "FRZB"],
    "Endothelial":    ["ESAM", "FLT1", "RAMP2", "VWF", "EGFL7", "SLC9A3R2", "CAVIN2", "ADGRL4", "ABLIM1", "ABCB1"],
    # --- Myeloid --- 
    # v3 改动: 强化 Macrophage (加 C1Q/APOE/CD163), 缩小 CDC (去 HLA-DPB1), 加 Microglia homeostatic
    "Monocyte":       ["FCN1", "VCAN", "S100A8", "S100A9", "CD14"],
    "Microglia":      ["TMEM119", "CX3CR1", "P2RY12", "TREM2", "GPR34", "CTSD", "CSF1R"],
    "Macrophage":     ["CD68", "LYZ", "C1QA", "C1QB", "C1QC", "APOE", "CD163", "MRC1", "GPNMB", "SPP1"],
    "CDC":            ["FCER1A", "CD1C", "CLEC9A", "LAMP3", "IRF8"],
    "Neutrophil":     ["S100A8", "FCGR3B", "IFITM2", "CSF3R", "CXCR2"],
}

# 过滤缺失基因
markers_filt = {}
for ct, genes in gbm_markers.items():
    found = [g for g in genes if g in adata.var_names]
    markers_filt[ct] = found
    missing = set(genes) - set(found)
    if missing:
        print(f'⚠️ {ct}: 缺失 {missing}')

print(f'\n共 {sum(len(v) for v in markers_filt.values())} 个 marker 基因可用')

In [ ]:
# Dotplot: v2 marker × leiden cluster
sc.pl.dotplot(adata, markers_filt, groupby='leiden',
              standard_scale='var', dendrogram=True,
              save='_gbm_markers_v2_by_leiden.png')

### 粗分类注释 (Coarse annotation)

In [ ]:
# 严格按照用户分类打分
gbm_markers_v3 = {
    # --- Tumor & Glia ---
    "Tumor":          ["GFAP", "PTPRZ1", "SCG3", "C1orf61", "SOX2", "SOX6", "BCAN", "NOVA1", "NRCAM", "TUBB2B"],
    "Prol.tumor":     ["SOX2", "SOX6", "MKI67", "TOP2A", "CCDC34", "PBK", "UBE2T", "CENPV", "CENPF"],
    "Oligodendrocyte":["MAG", "CLDN11", "APLP1", "TMEM144", "CNDP1", "EDIL3", "LARP6", "MOG", "AMER2", "TUBB4A"],
    "Neuronal":       ["SYT1", "STMN2", "RBFOX3", "GRIN1", "SLC17A7", "GAD1", "GAD2"],
    # --- Lymphoid ---
    "NK/T":           ["CD3D", "CD3E", "TRBC2", "CD3G", "CD2", "CD52", "TRAC", "IL7R", "TRBC1", "LCK", "SKAP1", "CD48"],
    # --- Vascular / Stromal ---
    "Pericyte":       ["COL1A2", "COL6A2", "PDGFRB", "COL3A1", "EDNRA", "LUM", "COL1A1", "PLAC9", "FRZB"],
    "Endothelial":    ["ESAM", "FLT1", "RAMP2", "VWF", "EGFL7", "SLC9A3R2", "CAVIN2", "ADGRL4", "ABLIM1", "ABCB1"],
    # --- Myeloid & Dendritic ---
    "Monocyte":       ["FCN1", "VCAN", "CD52", "S100A8", "S100A9"],
    "Microglia":      ["TMEM119", "CX3CR1", "TREM2", "GPNMB", "CTSD", "CD68"],
    "Macrophage":     ["LYZ", "FN1", "VCAN"],
    "CDC":            ["FCER1A", "HLA-DPB1", "CD83", "AREG"],
    "Neutrophil":     ["S100A8", "FCGR3B", "IFITM2"],
}

# 打分
for ct, genes in gbm_markers_v3.items():
    valid = [g for g in genes if g in adata.var_names]
    sc.tl.score_genes(adata, gene_list=valid, score_name=f'score_{ct}')
    missing = set(genes) - set(valid)
    status = f"({len(valid)}/{len(genes)})"
    if missing:
        status += f" missing: {missing}"
    print(f"  {ct}: {status}")

# 每个 cluster 的平均得分
score_cols = [f'score_{ct}' for ct in gbm_markers_v3]
df_v3 = adata.obs.groupby('leiden', observed=False)[score_cols].mean()
df_v3.columns = list(gbm_markers_v3.keys())

# 显示 + 自动分配
assign_v3 = df_v3.idxmax(axis=1).to_dict()
print("\n按最高分自动分配:")
for cl in sorted(assign_v3, key=int):
    scores = df_v3.loc[cl]
    top = scores.max()
    second = scores.nlargest(2).iloc[1]
    flag = " ⚠️" if (top - second) < 0.1 else ""
    print(f"  Cluster {cl:>2s} → {assign_v3[cl]:<16s} (top={top:.3f}, 2nd={scores.nlargest(2).index[1]}={second:.3f}){flag}")

print(f"\n得分矩阵:")
print(df_v3.round(3).to_string())

In [ ]:
# 查看分配结果
for cl in sorted(assign_v3, key=int):
    scores = df_v3.loc[cl]
    top = scores.max()
    second = scores.nlargest(2).iloc[1]
    flag = " ⚠️" if (top - second) < 0.1 else ""
    print(f"  Cluster {cl:>2s} → {assign_v3[cl]:<16s} (top={top:.3f}, 2nd={scores.nlargest(2).index[1]}={second:.3f}){flag}")

In [ ]:
# UMAP: 修正后注释 vs leiden
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sc.pl.umap(adata, color='cell_type', ax=axes[0],
           title='Cell type annotation (fixed)', show=False, frameon=False,
           legend_loc='on data', legend_fontsize=8)
sc.pl.umap(adata, color='leiden', ax=axes[1],
           title='Leiden clusters', show=False, frameon=False,
           legend_loc='on data', legend_fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'umap_celltype_annotation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 手动修正: 依据 v3 dotplot + Wilcoxon top markers
manual_override = {
    '0': 'Tumor',
    '1': 'Prol.tumor',    # TOP2A, CENPF, RRM2, ASPM → 增殖肿瘤
    '2': 'Macrophage',    # JUN, FOS, CCL3, CD83 → 活化 TAM
    '4': 'Macrophage',    # CD83, OLR1, KLF6, CCL3, CD74 → 活化 TAM
    '5': 'Macrophage',    # C1QB, C1QA, C1QC, HLA-DRA, CD74 → 补体型 TAM
    '7': 'Macrophage',    # MRC1, MERTK, CSF1R, STAB1, TLR2 → M2 型 TAM
    '10': 'Microglia',    # HSPs 为主 (应激), UMAP 靠近 cluster 0
}

# 应用修正
assign_fixed = assign_v3.copy()
assign_fixed.update(manual_override)

adata.obs['cell_type'] = adata.obs['leiden'].map(assign_fixed).astype('category')
print(adata.obs['cell_type'].value_counts())

In [ ]:
from sklearn.neighbors import KernelDensity

def _repel_labels(positions, min_dist=1.5, iterations=50):
    """简单标签排斥: 让过近的标签互相推开"""
    pos = np.array(positions, dtype=float)
    for _ in range(iterations):
        moved = False
        for i in range(len(pos)):
            for j in range(i + 1, len(pos)):
                d = pos[j] - pos[i]
                dist = np.linalg.norm(d)
                if dist < min_dist and dist > 0:
                    shift = (min_dist - dist) / 2 * d / dist
                    pos[i] -= shift
                    pos[j] += shift
                    moved = True
        if not moved:
            break
    return pos

def enhance_umap(adata, color='cell_type', figsize=(8, 8), size=15,
                 kde_bandwidth=0.3, contour_level_frac=0.1,
                 label_fontsize=10, arrow_frac=0.10, palette=None,
                 label_min_dist=2.0, show_pct=True, show_arrows=True,
                 contour_color='gray', contour_style='--',
                 title=None):
    """融合: 生信大杂烩 + 生信星球 UMAP 美化"""
    fig, ax = plt.subplots(figsize=figsize)

    sc.pl.umap(adata, color=color, legend_loc=None, frameon=False,
               size=size, ax=ax, add_outline=True, palette=palette,
               outline_width=(0.1, 0.05), show=False)

    umap_coords = adata.obsm['X_umap']
    categories = adata.obs[color].cat.categories
    colors_list = adata.uns.get(f'{color}_colors', [f'C{i}' for i in range(len(categories))])

    # 细胞比例
    ct_counts = adata.obs[color].value_counts(normalize=True) * 100

    # 收集标签位置
    label_info = []
    for i, ct in enumerate(categories):
        mask = adata.obs[color] == ct
        coords = umap_coords[mask]
        if len(coords) < 30:
            continue
        mx, my = np.median(coords, axis=0)
        label_info.append((ct, mx, my, i, coords))

    # 标签排斥
    raw_positions = [(info[1], info[2]) for info in label_info]
    adjusted = _repel_labels(raw_positions, min_dist=label_min_dist)

    # KDE 轮廓 + 标签
    for idx, (ct, _, _, ci, coords) in enumerate(label_info):
        kde = KernelDensity(bandwidth=kde_bandwidth, metric='euclidean')
        kde.fit(coords)
        x_min_ct, x_max_ct = coords[:, 0].min() - 0.5, coords[:, 0].max() + 0.5
        y_min_ct, y_max_ct = coords[:, 1].min() - 0.5, coords[:, 1].max() + 0.5
        xx, yy = np.mgrid[x_min_ct:x_max_ct:100j, y_min_ct:y_max_ct:100j]
        xy_test = np.vstack([xx.ravel(), yy.ravel()]).T
        Z = np.exp(kde.score_samples(xy_test)).reshape(xx.shape)
        peak = Z.max()
        if peak > 0:
            ax.contour(xx, yy, Z, levels=[peak * contour_level_frac],
                       colors=contour_color, linestyles=contour_style,
                       linewidths=1.2, alpha=0.8)

        # 标签 (排斥后位置 + 可选百分比)
        lx, ly = adjusted[idx]
        pct = ct_counts.get(ct, 0)
        label_text = f"{ct} ({pct:.1f}%)" if show_pct else ct
        ax.text(lx, ly, label_text, fontsize=label_fontsize,
                ha='center', va='center', color='black',
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

    # 科研风箭头坐标轴
    if show_arrows:
        x_all, y_all = umap_coords[:, 0], umap_coords[:, 1]
        x_lo, x_hi = x_all.min(), x_all.max()
        y_lo, y_hi = y_all.min(), y_all.max()
        x_range = x_hi - x_lo
        y_range = y_hi - y_lo
        ox = x_lo - x_range * 0.02
        oy = y_lo - y_range * 0.02
        dx = x_range * arrow_frac
        dy = y_range * arrow_frac
        ax.annotate('', xy=(ox + dx, oy), xytext=(ox, oy),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
        ax.annotate('', xy=(ox, oy + dy), xytext=(ox, oy),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
        ax.text(ox + dx / 2, oy - y_range * 0.03, 'UMAP1',
                fontsize=10, ha='center', va='top')
        ax.text(ox - x_range * 0.03, oy + dy / 2, 'UMAP2',
                fontsize=10, ha='right', va='center', rotation=90)

    # 标题
    if title is None:
        title = f"Overall (n = {adata.n_obs:,})"
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout()
    return fig, ax

print('enhance_umap 函数已定义 (融合版)')

In [ ]:
# 修正后 UMAP (融合美化版: KDE虚线轮廓 + 百分比标签 + 箭头坐标轴)
fig, ax = enhance_umap(adata, color='cell_type', figsize=(10, 10), size=15,
                       kde_bandwidth=0.3, label_fontsize=9,
                       label_min_dist=2.5, show_pct=True, show_arrows=True)
plt.savefig(FIG_DIR / 'umap_celltype_enhanced.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# doublet_score 分布直方图 + 阈值线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1) 全局 doublet_score 分布
axes[0].hist(adata.obs['doublet_score'], bins=100, color='steelblue', edgecolor='white')
for thr in [0.15, 0.20, 0.25]:
    n_above = (adata.obs['doublet_score'] > thr).sum()
    axes[0].axvline(thr, color='red', ls='--', lw=1.2)
    axes[0].text(thr + 0.002, axes[0].get_ylim()[1] * 0.9,
                 f'{thr} ({n_above:,})', fontsize=9, color='red')
axes[0].set_xlabel('doublet_score')
axes[0].set_ylabel('cells')
axes[0].set_title('Doublet Score Distribution')

# 2) 按 cell_type 的 doublet_score 箱线图
import seaborn as sns
order = adata.obs['cell_type'].value_counts().index.tolist()
sns.boxplot(data=adata.obs, x='cell_type', y='doublet_score', ax=axes[1], order=order)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].set_title('Doublet Score by Cell Type')
plt.tight_layout()
plt.savefig(FIG_DIR / 'doublet_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 检查 cluster 2 详细得分
print("Cluster 2 所有类型得分 (降序):")
scores_c2 = df_v3.loc['2'].sort_values(ascending=False)
for ct, s in scores_c2.items():
    print(f"  {ct:<16s}: {s:.3f}")

# 对比 cluster 1 和 2
print("\nCluster 1 vs 2 对比:")
for ct in ['Macrophage', 'Microglia', 'Monocyte', 'CDC', 'Neutrophil', 'Tumor']:
    print(f"  {ct:<16s}: cl1={df_v3.loc['1', ct]:.3f}  cl2={df_v3.loc['2', ct]:.3f}")

# 直接看 cluster 2 中 Macrophage/Microglia/Monocyte 关键 marker 的表达
key_check = ['CD68', 'LYZ', 'FN1', 'VCAN', 'TMEM119', 'CX3CR1',
             'TREM2', 'GPNMB', 'FCN1', 'S100A8', 'FCER1A', 'HLA-DPB1']
key_valid = [g for g in key_check if g in adata.var_names]

import numpy as np
c2_mask = adata.obs['leiden'] == '2'
c1_mask = adata.obs['leiden'] == '1'
print(f"\n关键 marker 平均表达 (cluster 1 vs 2):")
for g in key_valid:
    idx = adata.var_names.get_loc(g)
    expr_c1 = np.asarray(adata[c1_mask, idx].X.todense()).mean()
    expr_c2 = np.asarray(adata[c2_mask, idx].X.todense()).mean()
    print(f"  {g:<12s}: cl1={expr_c1:.3f}  cl2={expr_c2:.3f}")

## 3. Rank Genes Groups (Wilcoxon)

In [ ]:
import gc

# 排除核糖体/线粒体/泛表达 lncRNA
exclude = (
    adata.var_names.str.startswith(('RPS', 'RPL', 'MT-')) |
    adata.var_names.isin(['MALAT1', 'NEAT1'])
)
print(f'排除 {exclude.sum()} 个基因')

import anndata as ad
adata_clean = ad.AnnData(
    X=adata.X[:, ~exclude],
    obs=adata.obs[['leiden']].copy(),
    var=adata.var[~exclude][[]].copy(),
)
adata_clean.var_names = adata.var_names[~exclude].tolist()
print(f'过滤后: {adata_clean.shape[1]} 基因')

sc.tl.rank_genes_groups(adata_clean, groupby='leiden', method='wilcoxon')
adata.uns['rank_genes_groups'] = adata_clean.uns['rank_genes_groups']
del adata_clean; gc.collect()
print('✅ Wilcoxon rank_genes_groups 完成')

In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=10, save='_rank_genes_panels.pdf')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=4, standard_scale='var',
                                dendrogram=False,
                                save='_rank_genes_dotplot.pdf')

In [ ]:
# 每个 cluster top 5 marker 表格
top_markers = []
for cl in sorted(adata.obs['leiden'].cat.categories, key=int):
    df = sc.get.rank_genes_groups_df(adata, group=cl).head(5)
    df['cluster'] = cl
    top_markers.append(df)

top_df = pd.concat(top_markers)
top_df[['cluster', 'names', 'logfoldchanges', 'pvals_adj', 'scores']].to_csv(
    FIG_DIR / 'top5_markers_per_cluster.csv', index=False)
print(top_df.pivot_table(index='cluster', columns=top_df.groupby('cluster').cumcount(),
                         values='names', aggfunc='first').to_string())

## 6. 保存 adata2.h5ad

In [ ]:
# Wilcoxon on cell_type
import gc

exclude = (
    adata.var_names.str.startswith(('RPS', 'RPL', 'MT-')) |
    adata.var_names.isin(['MALAT1', 'NEAT1'])
)
adata_clean = ad.AnnData(
    X=adata.X[:, ~exclude],
    obs=adata.obs[['cell_type']].copy(),
    var=adata.var[~exclude][[]].copy(),
)
adata_clean.var_names = adata.var_names[~exclude].tolist()

sc.tl.rank_genes_groups(adata_clean, groupby='cell_type', method='wilcoxon')
adata.uns['rank_genes_groups_celltype'] = adata_clean.uns['rank_genes_groups']
del adata_clean; gc.collect()
print('✅ cell_type Wilcoxon 完成')

In [ ]:
# dendrogram
sc.tl.dendrogram(adata, 'cell_type', use_rep='X_pca_harmony', n_pcs=30)

# 保存
out_path = WRITE_DIR / 'adata2.h5ad'
adata.write(out_path)
print(f'✅ 已保存: {out_path}')
print(f'   {adata.shape[0]} cells × {adata.shape[1]} genes')
print(f'   cell_type: {adata.obs["cell_type"].nunique()} 类')
adata